### 4.6.1 贝叶斯计算

假设观测数据$\boldsymbol{y}$由抽样模型$f(\boldsymbol{y}|\boldsymbol{\theta})$生成,其中$\boldsymbol{\theta}$为未知参数组.在经典统计学中,通过最大化似然函数得到参数估计量,即$\widehat{\boldsymbol{\theta}}=\argmax_{\boldsymbol{\theta}}f(\boldsymbol{y}|\boldsymbol{\theta})$;随后通过从抽样模型中反复生成$\boldsymbol{y}$样本,评估估计量$\widehat{\boldsymbol{\theta}}$的不确定性.在贝叶斯统计学框架下,我们先设定先验分布$f(\boldsymbol{\theta})$,用以刻画获取数据前参数$\boldsymbol{\theta}$的不确定性;再借助贝叶斯定理得到更新后的分布,也就是后验分布(假定参数$\boldsymbol{\theta}$为连续型变量):

$$f(\boldsymbol{\theta}|\boldsymbol{y})=\frac{f(\boldsymbol{y}|\boldsymbol{\theta})f(\boldsymbol{\theta})}{\int f(\boldsymbol{y}|\boldsymbol{\theta})f(\boldsymbol{\theta})\mathrm{d}\boldsymbol{\theta}}$$

公式分母为归一化常数,通常是高维积分,难以直接求解.因此一般采用马尔可夫链蒙特卡洛方法,从后验分布$f(\boldsymbol{\theta}|\boldsymbol{y})$中抽取若干参数样本完成推断,该方法无需计算归一化常数.

马尔可夫链蒙特卡洛方法通常需要大量样本(数千甚至上百万个),才能较好近似后验分布.倘若非归一化后验$h(\boldsymbol{\theta})=f(\boldsymbol{y}|\boldsymbol{\theta})f(\boldsymbol{\theta})$的求值成本很高,采样过程将十分耗时.针对这类情形,可以采用试验设计方法选取采样点,对非归一化后验进行求值,并利用高斯过程模型拟合近似(约瑟夫,2012,2013).

如何构造适用于后验近似的试验设计?显而易见,我们应当在后验分布的高密度区域布置更多采样点,在后验低密度区域少布置甚至不布置采样点.最小能量设计恰好契合这一思路,此时电荷函数取$q(\boldsymbol{x})=f(\boldsymbol{\theta}|\boldsymbol{y})^{-1/(2p)}$.由此可得:

$$\begin{align*}D^*&=\mathop{\argmax}_{D}\;\min_{i\ne j}f(\boldsymbol{\theta}_i|\boldsymbol{y})f(\boldsymbol{\theta}_j|\boldsymbol{y})\|\boldsymbol{\theta}_i-\boldsymbol{\theta}_j\|_{\alpha}^{2p}\\&=\mathop{\argmax}_{D}\;\min_{i\ne j}h(\boldsymbol{\theta}_i)h(\boldsymbol{\theta}_j)\|\boldsymbol{\theta}_i-\boldsymbol{\theta}_j\|_{\alpha}^{2p}\\&=\mathop{\argmax}_{D}\;\min_{i\ne j}(\log h(\boldsymbol{\theta}_i)+\log h(\boldsymbol{\theta}_j)+2p\log\|\boldsymbol{\theta}_i-\boldsymbol{\theta}_j\|_{\alpha})\tag{4.47}\end{align*}$$

可以注意到,归一化常数在优化过程中被分离出去.也就是说,求解最小能量设计时不需要用到归一化常数,这是一大优势.上式最后一步取对数,是因为非归一化后验的取值可能极小,直接计算容易引发数值误差.

上述公式存在一处缺陷:对式(4.47)开展优化,需要反复计算非归一化后验,这与我们借助试验设计近似高代价后验的初衷相悖.约瑟夫等人(2019)提出了一种高效算法,仅需少量非归一化后验求值,就能生成最小能量设计.该算法核心思路是采用退火形式的非归一化后验密度:

$$h^{\gamma}(\boldsymbol{\theta}),\;\gamma\in[0,1]\tag{4.48}$$

当$\gamma=1$时,该分布就是目标后验分布;当$\gamma=0$时,得到定义在后验支撑集上的均匀分布.假设经过合理尺度变换,后验的支撑集可写为$\mathcal{X}=[0,1]^p$.我们可以先在区域$[0,1]^p$上构造初始设计,随后缓慢将$\gamma$增大至1,同时迭代学习非归一化后验.

举个例子,哈尔里奥等人(1999)给出了形如香蕉状的密度函数:

$$f(\boldsymbol{\theta})\propto\exp\left(-\frac{1}{2}\frac{\theta_1^2}{100}-\frac{1}{2}(\theta_2+0.03\theta_1^2-3)^2\right),\;\boldsymbol{\theta}\in\mathbb{R}^2\tag{4.49}$$

该后验本身计算成本不高,此处仅作演示.假定已知后验高密度区域落在区间$[-40,40]\times[-25,10]$内,我们将该矩形区域缩放至单位正方形$[0,1]^2$.首先在正方形内构造包含20个样本点的最大投影设计.图4.24左图展示了这20个样本点,可以看到最大投影设计完全没有覆盖后验的高密度区域.接下来,我们基于式(4.46)的序贯算法,令$\gamma=1/4$,同时用局部近似$\widehat{h}(\cdot)$替换原函数$h(\cdot)$,新增20个最小能量设计采样点.随后依次将$\gamma$调整为$2/4,3/4$,最终取$\gamma=1$,重复上述过程.该算法在R语言程序包`mined`中实现(王,约瑟夫,2022).最终生成100个最小能量设计采样点,结果如图4.24右图所示.不难看出,采样点集中分布在后验高密度区域,是用于近似后验的优良试验设计.

> **图4.24** 左图展示初始20个最大投影设计点(红色),叠加绘制在香蕉形后验分布图像之上.右图展示通过退火策略分四步依次新增的80个最小能量设计点(蓝色)

In [10]:
#图4.24

options(repr.plot.width=20,repr.plot.height=10)
par(mfrow=c(1,2))
logf=function(para){
  l1=-40;u1=40;l2=-25;u2=10
  x1=l1+(u1-l1)*para[1]
  x2=l2+(u2-l2)*para[2]
  -.5*(x1^2/100+(x2+.03*x1^2-3)^2)
}
N.plot=300
p1=seq(0,1,length.out=N.plot)
p2=seq(0,1,length.out=N.plot)
fc=matrix(0,nrow=N.plot,ncol=N.plot)
for(i in 1:N.plot)for(j in 1:N.plot)fc[i,j]=exp(logf(c(p1[i],p2[j])))
library(fields)
imagePlot(p1,p2,fc,xlab=expression(theta[1]),ylab=expression(theta[2]),col=cm.colors(5),main="初始最大投影设计",cex.main=3,cex.lab=2,cex.axis=2)
p=2;n=20
set.seed(8)
library(MaxPro)
ini=MaxPro(MaxProLHD(n,p)$Design)$Design
points(ini,pch=16,col="red",cex=3)

imagePlot(p1,p2,fc,xlab=expression(theta[1]),ylab=expression(theta[2]),col=cm.colors(5),main="退火策略下的MED点",cex.main=3,cex.lab=2,cex.axis=2)
points(ini,pch=16,col="red",cex=3)
library(mined)
cand=mined(ini,logf,K_iter=5)$cand
ind=match(ini[,1],cand[,1])
points(cand[-ind,],pch=16,col="blue",cex=3)
par(mfrow=c(1,1))

下面介绍约瑟夫(2013)提出的基于试验设计的插值方法,用以近似后验分布.设设计点集$D=\{\boldsymbol{\nu}_1,\dots,\boldsymbol{\nu}_n\}$,$h_i=h(\boldsymbol{\nu}_i),\;i=1,\dots,n$.我们通过基函数展开近似非归一化密度的平方根:

$$\sqrt{h(\boldsymbol{\theta})}\approx\sum_{i=1}^mc_ig(\boldsymbol{\theta};\boldsymbol{\nu}_i,\boldsymbol{\Sigma})\tag{4.50}$$

对密度取平方根,是为了保证密度预测结果非负.其中

$$g(\boldsymbol{\theta};\boldsymbol{\mu},\boldsymbol{\Sigma})=\exp\left(-\frac{1}{2}\sum_{i=1}^p\frac{(\theta_i-\mu_i)^2}{\sigma_i^2}\right)$$

$\boldsymbol{\Sigma}=\mathrm{diag}\{\sigma_1^2,\dots,\sigma_p^2\}$.参照2.1.2节的方法,可通过交叉验证估计参数$\sigma_i^2$.将表达式关于所有$\theta_i$在$(-\infty,+\infty)$积分,即可求得归一化常数:

$$\begin{align*}\int\widehat{h}(\boldsymbol{\theta})\mathrm{d}\boldsymbol{\theta}&=\int\widehat{\boldsymbol{c}}'\boldsymbol{g}(\boldsymbol{\theta})\boldsymbol{g}(\boldsymbol{\theta})'\widehat{\boldsymbol{c}}\,\mathrm{d}\boldsymbol{\theta}\\&=\pi^{d/2}|\boldsymbol{\Sigma}|^{1/2}\widehat{\boldsymbol{c}}'\boldsymbol{G}_2\widehat{\boldsymbol{c}}\tag{4.51}\end{align*}$$

式中$\boldsymbol{G}_2$是$n\times n$矩阵,矩阵第$i$行第$j$列元素为$g(\boldsymbol{\nu}_i;\boldsymbol{\nu}_j,2\boldsymbol{\Sigma})$.由此得到后验密度近似表达式:

$$\widehat{f}(\boldsymbol{\theta}|\boldsymbol{y})=\frac{\{\widehat{\boldsymbol{c}}'\boldsymbol{g}(\boldsymbol{\theta})\}^2}{\pi^{d/2}|\boldsymbol{\Sigma}|^{1/2}\widehat{\boldsymbol{c}}'\boldsymbol{G}_2\widehat{\boldsymbol{c}}}\tag{4.52}$$

基于上式可以推导出边缘密度:

$$\widehat{p}(\theta_k|\boldsymbol{y})=\frac{\sum_{i=1}^n\sum_{j=1}^nd_{ij}\varphi(\theta_k;(\nu_{ik}+\nu_{jk})/2,\Sigma_{kk}/2)}{\sum_{i=1}^n\sum_{j=1}^nd_{ij}}\tag{4.53}$$

其中$\varphi(\cdot;\mu,\sigma^2)$代表均值为$\mu$,方差为$\sigma^2$的正态密度函数,$d_{ij}=\widehat{c}_i\widehat{c}_jg(\boldsymbol{\nu}_i;\boldsymbol{\nu}_j,2\boldsymbol{\Sigma})$.

该插值方法作用于$\sqrt{h(\boldsymbol{\theta})}$,我们以$\sqrt{h(\boldsymbol{\theta})}$为目标密度,生成如图4.24所示包含100个采样点的最小能量设计.采用该方法对式(4.49)中香蕉状密度求解得到$\theta_1,\theta_2$的边缘密度,结果见图4.25.本例中真实后验可以通过数值积分算出,对应图中绿色曲线.作为对比,图中红色曲线为借助维霍拉(2012)的自适应梅特罗波利斯算法生成一万条马尔可夫链蒙特卡洛样本得到的密度曲线.不难看出,该插值方法对真实后验的近似效果远优于马尔可夫链蒙特卡洛方法.本例中插值方法仅需要100次函数求值,而马尔可夫链蒙特卡洛方法需要一万次求值.当面对计算成本极高的后验分布时,这一优势将十分突出.

> **图4.25** 香蕉形后验分布的边际后验密度:数值积分法(绿色),基于一百个点最大熵设计的推断算法(蓝色),包含一万个样本的自适应梅特罗波利斯算法(红色)

In [2]:
#图4.25

logf=function(para){
  x1=-40+80*para[1]
  x2=-25+35*para[2]
  -.5*(x1^2/100+(x2+.03*x1^2-3)^2)
}
h=function(para)exp(logf(para))
N.plot=300
p1=seq(0,1,l=N.plot)
p2=seq(0,1,l=N.plot)
p=2;n=20
library(cubature)
denom=adaptIntegrate(h,c(0,0),c(1,1))$int
exact=matrix(0,nrow=N.plot,ncol=p)
for(i in 1:N.plot){
  theta2=p2[i]
  exact[i,2]=integrate(function(t1)apply(cbind(t1,theta2),1,h),0,1)$val/denom
}
for(i in 1:N.plot){
  theta1=p1[i]
  exact[i,1]=integrate(function(t2)apply(cbind(theta1,t2),1,h),0,1)$val/denom
}

library(adaptMCMC)
theta=MCMC(logf,n=10000,init=rep(.5,p),scale=(2.4/sqrt(2))^2*diag(p),adapt=TRUE,acc.rate=.05)$samples

set.seed(8)
library(MaxPro)
ini=MaxPro(MaxProLHD(n,p)$Design)$Design
library(mined)
nu=mined(ini,logf,K_iter=5)$cand

library(sqpost)
hev=apply(nu,1,h)
fit=sqrt_fit(nu,hev)
den=cbind(marginal(fit,p1,dim=1),marginal(fit,p2,dim=2))

options(repr.plot.width=20,repr.plot.height=10)
par(mfrow=c(1,2))
u=max(c(den[,1],exact[,1]))*(1.2-.2*(1-1))
plot(density(theta[,1]),type="l",col=2,lwd=4,ylim=c(0,u),main="",xlab=expression(theta[1]),ylab="密度",cex.lab=2,cex.axis=2)
lines(p1,exact[,1],lwd=4,col=3)
lines(p1,den[,1],col="blue",lwd=4)
u=max(c(den[,2],exact[,2]))*(1.2-.2*(2-1))
plot(density(theta[,2]),type="l",col=2,lwd=4,ylim=c(0,u),main="",xlab=expression(theta[2]),ylab="密度",cex.lab=2,cex.axis=2)
lines(p2,exact[,2],lwd=4,col=3)
lines(p2,den[,2],col="blue",lwd=4)
legend(-1.4,7,legend=c("真实值","插值方法","马尔可夫链\n蒙特卡洛方法"),col=c(3,"blue",2),lwd=c(4,4,4),bty="n",cex=2)
par(mfrow=c(1,1))

  generate 10000 samples 
